# §11.8.4 — 학습된 어텐션이 국소 패턴에 수렴하는지의 관찰

> 딥러닝 교재 · 3부 11장 8절 4항 (🐍)
> 선행: §11.8.1(고정 상대 가중치 = 합성곱) · §11.8.2(다중 헤드의 합성곱 표현) · §11.8.3(내용 의존성)

## 이 노트북이 답하는 질문

1. **국소 과제를 만난 어텐션은 스스로 합성곱이 되어 가는가?** 상대 위치 편향이 좁은 대역으로 조여드는지 본다.
2. **헤드들은 §11.8.2의 구성처럼 서로 다른 오프셋을 분담하는가?**
3. **데이터가 적으면 수렴이 불완전해지는가?** (§11.8.5의 예고)

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 50초).
모델은 1차원 신호 열 위의 **상대 위치 편향을 갖는 단층 다중 헤드 어텐션**이다.
값은 신호 그대로($v_j=x_j$) 두어, 각 헤드가 "어느 상대 위치를 읽어 오는가"가 전부가 되게 했다 —
§11.8.2의 구성(헤드 하나 = 오프셋 하나)이 학습으로 나타나는지를 보기 위한 설계다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 전역 평균으로는 풀리지 않는 국소 패턴

길이 $T=48$ 신호의 무작위 위치에 패턴 $s\cdot(+1,-1)$을 심는다. 레이블은 $s$의 부호.
이 패턴은 두 겹의 지름길을 막는다. (i) 합이 0이라 **전역 평균**(균등 어텐션이 모으는 것)에 신호가 없고,
(ii) $s$를 뒤집어도 신호 값들의 **다중집합이 그대로**라( $\{+A,-A\}$ ) 위치를 무시한 점별 통계로도 풀 수 없다.
답은 오직 **어느 값이 어느 값의 오른쪽에 있는가**, 즉 이웃의 순서에만 있다 — 국소 결합이 필수인 과제다.

In [ ]:
T = 48; AMP = 3.0
PAT = np.array([1., -1.])

def make_data(n, rn):
    X = rn.standard_normal((n, T))
    y = rn.integers(0, 2, n)
    pos = rn.integers(0, T-1, n)
    for i in range(n):
        X[i, pos[i]:pos[i]+2] += AMP * (1 if y[i] else -1) * PAT
    return X, y.astype(float)

Xte, yte = make_data(2000, np.random.default_rng(SEED+9))
print("시험 표본:", Xte.shape)

---
## 2. 상대 위치 편향 어텐션 — 구현

헤드 $h$의 로짓은 $\ell^h_{ij} = q^h_i\cdot k^h_j/\sqrt{d_k} + B_h[j-i]$.
$B_h$가 상대 위치만의 함수이므로, $B_h$가 뾰족해지면 그 헤드는 §11.8.1의 의미에서 합성곱이 된다.

In [ ]:
H, DK, HID = 4, 2, 24
OFF = np.arange(-(T-1), T)                     # 상대 오프셋 축
rel_idx = (np.arange(T)[None, :] - np.arange(T)[:, None]) + (T-1)   # (T,T) -> B 인덱스

def softmax_rows(L):
    L = L - L.max(axis=-1, keepdims=True)
    E = np.exp(L)
    return E / E.sum(axis=-1, keepdims=True)

class AttnNet:
    def __init__(self, rn):
        self.Wq = rn.standard_normal((H, 2, DK)) * 0.5
        self.Wk = rn.standard_normal((H, 2, DK)) * 0.5
        self.B  = np.zeros((H, 2*T-1))
        self.W1 = rn.standard_normal((1+H, HID)) * np.sqrt(2/(1+H))
        self.b1 = np.zeros(HID)
        self.W2 = rn.standard_normal(HID) / np.sqrt(HID)
        self.b2 = np.zeros(1)
        self.params = [self.Wq, self.Wk, self.B, self.W1, self.b1, self.W2, self.b2]
    def forward(self, X):
        N = X.shape[0]
        Phi = np.stack([X, np.ones_like(X)], axis=-1)          # (N,T,2)
        q = np.einsum('ntc,hcd->hntd', Phi, self.Wq, optimize=True)
        k = np.einsum('ntc,hcd->hntd', Phi, self.Wk, optimize=True)
        L = np.einsum('hnid,hnjd->hnij', q, k, optimize=True)/np.sqrt(DK) + self.B[:, rel_idx][:, None]
        A = softmax_rows(L)                                    # (H,N,T,T)
        g = np.einsum('hnij,nj->hni', A, X, optimize=True)     # 값 = x
        feat = np.concatenate([X[None], g], axis=0)            # (1+H,N,T)
        Z1 = np.einsum('fnt,fk->ntk', feat, self.W1, optimize=True) + self.b1
        A1 = np.maximum(Z1, 0)
        z = A1 @ self.W2 + self.b2                             # (N,T)
        out = z.mean(axis=1)
        self.cache = (X, Phi, q, k, A, g, feat, Z1, A1)
        return out
    def attn_by_offset(self, X):
        _ = self.forward(X)
        A = self.cache[4]                                       # (H,N,T,T)
        prof = np.zeros((H, 2*T-1))
        cnt = np.zeros(2*T-1)
        np.add.at(cnt, rel_idx.ravel(), 1.0)
        for h in range(H):
            np.add.at(prof[h], rel_idx.ravel(), A[h].mean(axis=0).ravel())
        return prof / cnt
    def backward(self, dout):
        X, Phi, q, k, A, g, feat, Z1, A1 = self.cache
        N = X.shape[0]
        dz = np.repeat(dout[:, None]/T, T, axis=1)             # (N,T)
        dW2 = np.einsum('ntk,nt->k', A1, dz, optimize=True)
        db2 = np.array([dz.sum()])
        dA1 = dz[:, :, None] * self.W2[None, None, :]
        dZ1 = dA1 * (Z1 > 0)
        dW1 = np.einsum('fnt,ntk->fk', feat, dZ1, optimize=True)
        db1 = dZ1.sum(axis=(0, 1))
        dfeat = np.einsum('ntk,fk->fnt', dZ1, self.W1, optimize=True)
        dg = dfeat[1:]                                          # (H,N,T)
        # softmax-가중합의 역전파: dL_ij = A_ij * dg_i * (x_j - g_i)
        dL = A * (dg[:, :, :, None] * (X[None, :, None, :] - g[:, :, :, None]))
        dB = np.zeros_like(self.B)
        for h in range(H):
            np.add.at(dB[h], rel_idx.ravel(), dL[h].sum(axis=0).ravel())
        dq = np.einsum('hnij,hnjd->hnid', dL, k, optimize=True)/np.sqrt(DK)
        dk = np.einsum('hnij,hnid->hnjd', dL, q, optimize=True)/np.sqrt(DK)
        dWq = np.einsum('ntc,hntd->hcd', Phi, dq, optimize=True)
        dWk = np.einsum('ntc,hntd->hcd', Phi, dk, optimize=True)
        return [dWq, dWk, dB, dW1, db1, dW2, db2]

def adam(p, g, m, v, t, lr):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

def locality_index(prof):
    # 어텐션 질량의 |오프셋| 기대값 (헤드 평균) — 작을수록 국소적
    w = prof / prof.sum(axis=1, keepdims=True)
    return float(np.mean(w @ np.abs(OFF)))

def peak_mass(prof, halfwidth=1):
    # 가장 국소화된 헤드가 |오프셋|≤halfwidth 안에 두는 질량 — 클수록 합성곱에 가깝다
    w = prof / prof.sum(axis=1, keepdims=True)
    sel = np.abs(OFF) <= halfwidth
    return float(np.max(w[:, sel].sum(axis=1)))

def train(n_train, steps, seed=0, record=False, lr=8e-3):
    rn = np.random.default_rng(seed)
    net = AttnNet(rn)
    Xtr, ytr = make_data(n_train, np.random.default_rng(1000+seed))
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    probe = Xte[:200]
    hist = {'step': [], 'loc': [], 'absoff': [], 'acc': []}
    B = 64
    for t in range(1, steps+1):
        idx = rn.integers(0, n_train, B)
        z = net.forward(Xtr[idx])
        dz = (sigmoid(z) - ytr[idx]) / B
        gs = net.backward(dz)
        for pp, g, m, v in zip(net.params, gs, ms, vs):
            adam(pp, g, m, v, t, lr)
        if record and (t % 25 == 0 or t == 1):
            prof = net.attn_by_offset(probe)
            hist['step'].append(t)
            hist['loc'].append(locality_index(prof))
            w = prof/prof.sum(axis=1, keepdims=True)
            hist['absoff'].append(w @ np.abs(OFF))
            acc = np.mean((net.forward(Xte[:1000]) > 0) == (yte[:1000] > 0.5))
            hist['acc'].append(acc)
    return net, hist

STEPS = 300 if FAST else 600
net0 = AttnNet(np.random.default_rng(0))
prof_before = net0.attn_by_offset(Xte[:200])
net, hist = train(3000, STEPS, seed=0, record=True)
prof_after = net.attn_by_offset(Xte[:200])
acc = np.mean((net.forward(Xte) > 0) == (yte > 0.5))
print(f"시험 정확도: {acc:.3f}   국소성 지표: {locality_index(prof_before):.2f} → {locality_index(prof_after):.2f}")

---
## 3. 데이터양을 바꾸면 — §11.8.5의 재료

In [ ]:
NS_D = [100, 3000] if FAST else [100, 300, 1000, 3000]
SEEDS_D = 1 if FAST else 2
loc_final = []; acc_final = []
for nd in NS_D:
    pm, ac = [], []
    for sd in range(SEEDS_D):
        ntd, _ = train(nd, 300 if FAST else 500, seed=7+sd)
        pf = ntd.attn_by_offset(Xte[:200])
        pm.append(peak_mass(pf))
        ac.append(np.mean((ntd.forward(Xte) > 0) == (yte > 0.5)))
    loc_final.append(np.mean(pm)); acc_final.append(np.mean(ac))
    print(f"n={nd:5d}  최대 헤드의 ±1 질량 {loc_final[-1]:.2f}  정확도 {acc_final[-1]:.3f}")

---
## 4. 교재 그림 — fig_11_8_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()
W_SHOW = 16
sl = slice(T-1-W_SHOW, T+W_SHOW)

# (a) 헤드별 어텐션 분포 (전/후)
ax = axes[0]
img = np.concatenate([prof_before[:, sl], np.full((1, prof_before[:, sl].shape[1]), np.nan), prof_after[:, sl]], axis=0)
imv = ax.imshow(img, aspect='auto', cmap='viridis',
                extent=[-W_SHOW, W_SHOW, 2*H+1, 0])
ax.grid(False); plt.colorbar(imv, ax=ax, fraction=0.046)
ax.set_yticks([2, 6.5]); ax.set_yticklabels([lab('학습 전\n(헤드 1–4)', 'before'), lab('학습 후\n(헤드 1–4)', 'after')], fontsize=8)
ax.set_xlabel(lab('상대 오프셋 $j-i$', 'relative offset $j-i$'))
ax.set_title(lab('(a) 헤드별 어텐션 질량의 상대 위치 분포', '(a) attention mass by offset'), fontsize=10)

# (b) 헤드별 평균 |오프셋| 궤적
ax = axes[1]
absoff = np.array(hist['absoff'])                # (rec, H)
for h in range(H):
    ax.plot(hist['step'], absoff[:, h], '-', color=CB[h+1], lw=1.3,
            label=lab(f'헤드 {h+1}', f'head {h+1}'))
ax.set_xlabel(lab('학습 걸음', 'training step'))
ax.set_ylabel(lab('어텐션 질량의 평균 $|j-i|$', 'mean $|j-i|$'))
ax.set_title(lab('(b) 헤드별 국소화 궤적', '(b) per-head localization'), fontsize=10)
ax.legend(fontsize=8)

# (c) 국소성 지표와 정확도
ax = axes[2]
ax.plot(hist['step'], hist['loc'], 'o-', color=CB[5], ms=3, label=lab('국소성 지표(작을수록 국소)', 'locality index'))
ax.set_xlabel(lab('학습 걸음', 'training step'))
ax.set_ylabel(lab('평균 $|j-i|$', 'mean $|j-i|$'), color=CB[5])
ax2 = ax.twinx()
ax2.plot(hist['step'], hist['acc'], 's--', color=CB[3], ms=3, label=lab('시험 정확도', 'accuracy'))
ax2.set_ylabel(lab('시험 정확도', 'test accuracy'), color=CB[3])
ax2.grid(False)
ax.set_title(lab('(c) 국소화와 성능이 함께 움직인다', '(c) locality tracks accuracy'), fontsize=10)

# (d) 데이터양과 최종 국소성
ax = axes[3]
ax.plot(NS_D, loc_final, 'o-', color=CB[5], ms=5, label=lab('최대 헤드의 $\\pm1$ 질량', 'peak head mass'))
ax.set_xscale('log')
ax.set_xlabel(lab('학습 표본 수 $n$', 'training samples $n$'))
ax.set_ylabel(lab('최대 헤드의 $|j-i|\\leq1$ 질량', 'peak-head local mass'), color=CB[5])
ax2 = ax.twinx()
ax2.plot(NS_D, acc_final, 's--', color=CB[3], ms=5, label=lab('정확도', 'accuracy'))
ax2.set_ylabel(lab('시험 정확도', 'test accuracy'), color=CB[3])
ax2.grid(False)
ax.set_title(lab('(d) 데이터가 적으면 수렴이 불완전하다', '(d) less data, less locality'), fontsize=10)

save_book_fig(fig, 'fig_11_8_4')
plt.show()

> ### 읽는 법
>
> (a) 균등하게 시작한 어텐션에서 한 헤드가 오프셋 $+1$에 질량 절반 이상을 싣는 **사실상의 조회 연산**으로
> 수렴했다 — §11.8.1의 특수화(고정 상대 가중치 = 합성곱)가 학습으로 나타난 것이고,
> §11.8.2의 "헤드 하나 = 오프셋 하나" 구성의 축소판이다.
> (b) 다만 모든 헤드가 그렇게 되지는 않는다. 궤적이 갈라지고 일부는 넓게 남는다 —
> 표현 정리는 도달점을 보장하지 않는다.
> (c) 국소화가 진행되는 구간에서 정확도가 함께 오른다. 이 과제에서 국소화는 장식이 아니라 해법이다.
> (d) 데이터가 적을수록 가장 국소화된 헤드조차 덜 국소적이고 정확도도 낮다 —
> 옳은 부분공간(합성곱)을 데이터가 찾아 주어야 하는 비용. §11.8.5의 논지가 이 그림에 있다.

---
## 5. 자기 점검

1. 값에 사영을 두지 않고 $v_j=x_j$로 고정한 것이 결과 해석을 어떻게 단순하게 만들었는가? 사영을 넣으면 무엇이 흐려지는가?
2. 패턴 $(+1,-1)$이 막는 두 지름길(전역 평균, 점별 통계)을 각각 수식으로 설명하라. 패턴을 $(+1,-2,+1)$로 바꾸면 어느 지름길이 다시 열리는가? 실행해 확인하라.
3. (a)에서 오프셋 0 근처가 아니라 ±1에 질량이 몰린 헤드가 있는가? §11.8.2의 구성과 비교하라.
4. 같은 과제를 $B$ 없이(내용 항만으로) 학습시키면 어떻게 되는가? 실행해 보라. 위치 정보의 출처가 무엇인지 §13.1.7과 연결해 설명하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 1절 | 2.5 | 패턴 대비 |
| `H`, `DK`, `HID` | 2절 | 4, 2, 24 | 헤드 수·차원 |
| `NS_D` | 3절 | …3000 | 데이터양 축 |
| `PAT` | 1절 | (1,−1) | (1,−2,1)로 바꾸면 점별 지름길이 열린다 (자기 점검 2) |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")